In [ ]:
#%config InlineBackend.figure_format = 'retina'
#%matplotlib inline
# Import necessary libraries
import os
import sys
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import scipy.stats as stats


In [ ]:
sns.set_context("talk", font_scale=1.5)

In [ ]:
# write a function to transform lat and lon to latitude and longitude, and convert longitude values from -180 to 180 to 0 to 360
def transform_coordinates(data):
    # if coordinates are lat and lon, rename them to latitude and longitude
    if 'lat' in data.coords:
        print("Renaming coordinates from 'lat' and 'lon' to 'latitude' and 'longitude'")
        data = data.rename({'lat': 'latitude', 'lon': 'longitude'})
    # if longitude values are from -180 to 180, convert them to 0 to 360
    if data.longitude.min() < 0:
        print("Converting longitude values from -180 to 180 to 0 to 360")
        data = data.assign_coords(longitude=(((data.longitude + 360) % 360)))
        # Sort the longitude values in ascending order
        data = data.sortby('longitude')
    return data

# Function to compute the compute climatology and anomaly
def compute_climatology_and_anomaly(data, climatology_period=(1981, 2010),study_period=(1981, 2020)):
    
    """
    Compute the climatology and anomaly of the input data.
    
    Parameters:
    data (xarray.DataArray): Input data to compute climatology and anomaly.
    climatology_period (tuple): Start and end year for the climatology period.
    study_period (tuple): Start and end year for the study period.
    Returns:
    tuple: Climatology and anomaly data.
    """

    # Select the study period
    data = data.sel(time=slice(f"{study_period[0]}-01-01", f"{study_period[1]}-12-31"))
    # Select the climatology period
    climatology_data = data.sel(time=slice(f"{climatology_period[0]}-01-01", f"{climatology_period[1]}-12-31"))
    
    # Compute the monthly climatology
    climatology = climatology_data.groupby('time.month').mean(dim='time')
    
    # Compute the anomaly by subtracting the climatology from the original data
    anomaly = data.groupby('time.month') - climatology
    
    return climatology, anomaly


# Function to detrend data
def detrend_data(data):
    """
    Detrend the input data by removing the linear trend.
    
    Parameters:
    data (xarray.DataArray): Input data to be detrended.
    
    Returns:
    xarray.DataArray: Detrended data.
    """
    coeff = data.polyfit(dim='time', deg=1)
    trend = xr.polyval(data['time'], coeff["polyfit_coefficients"])
    detrended_data = data - trend

    return detrended_data



# function to convert lon_min_lon_max in the 0-360 coordinate
def lon_min_lon_max(lon_min,lon_max):
    lon_min = (lon_min + 360)%360
    lon_max = (lon_max + 360)%360
    # order the longitudes
    #lon_min = min(lon1,lon2)
    #lon_max = max(lon1,lon2)
    return lon_min,lon_max

# function to select regions
def select_region(data, lon_min, lon_max, lat_min, lat_max):
    """
    Select a region from the input data based on the specified longitude and latitude bounds.
    
    Parameters:
    data (xarray.DataArray): Input data to select the region from.
    lon_min (float): Minimum longitude of the region.
    lon_max (float): Maximum longitude of the region.
    lat_min (float): Minimum latitude of the region.
    lat_max (float): Maximum latitude of the region.
    
    Returns:
    xarray.DataArray: Data for the selected region.
    """
    if lon_min < 0 and lon_max >= 0:
        lon_min = (lon_min + 360) % 360
        lon_max = (lon_max + 360) % 360
        region_data = data.where(((data.longitude >= lon_min) | (data.longitude <= lon_max)) & ((data.latitude >= lat_min) & (data.latitude <= lat_max)), drop=True)
    else:
        lon_min = (lon_min + 360) % 360
        lon_max = (lon_max + 360) % 360
        region_data = data.where(((data.longitude >= lon_min) & (data.longitude <= lon_max)) & ((data.latitude >= lat_min) & (data.latitude <= lat_max)), drop=True)


    #region_data = data.where(((data.longitude >= lon_min) | (data.longitude <= lon_max)) & ((data.latitude >= lat_min) & (data.latitude <= lat_max)), drop=True)
    
    return region_data,lon_min,lon_max


In [ ]:
import requests

url = "https://downloads.psl.noaa.gov/Datasets/noaa.ersst.v5/sst.mnmean.nc"

r = requests.get(url, stream=True)
r.raise_for_status()
dataname = 'ersst_v5'
filename = f"{dataname}.nc"
with open(filename, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024*1024):
        if chunk:
            f.write(chunk)

In [ ]:
dat = xr.open_dataset(filename, mask_and_scale=True)

In [ ]:
# drop time_bnds
dat = dat.drop_vars("time_bnds")
print(dat.lon.values[:5])


dat = transform_coordinates(dat)
#dat = mask_invalid_values(dat, vmin=-100, vmax=100)

In [ ]:
# Plot the last time step of the dataset
fig,ax = plt.subplots(figsize=(12,6),subplot_kw={'projection': ccrs.PlateCarree()})
dat.sst.isel(time=-1).plot(ax=ax,transform=ccrs.PlateCarree(),cmap='coolwarm',cbar_kwargs={'label': f"Sea Surface Temperature ({dat.sst.units})"})

In [ ]:
print(f"sst units: {dat.sst.units}")

In [ ]:
lons = dat.longitude.values
lats = dat.latitude.values

# prepare new lon and lat values, to interpolate the data to a new grid with 1 degree resolution
# first verify if lats are in ascending or descending order
tag_lat = 'ascending' if lats[0] < lats[-1] else 'descending'
print(f"Latitude values are in {tag_lat} order.")

# first verify if lons are in ascending or descending order
tag_lon = 'ascending' if lons[0] < lons[-1] else 'descending'
print(f"Longitude values are in {tag_lon} order.")

# prepare new lon and lat values, to interpolate the data to a new grid with 1 degree resolution
new_lons = np.arange(0, 360, 1) if tag_lon == 'ascending' else np.arange(359, -1, -1)
new_lats = np.arange(-90, 91, 1) if tag_lat == 'ascending' else np.arange(90, -91, -1)

# interpolate the data to a new grid with 1 degree resolution
dat = dat.interp(longitude=new_lons, latitude=new_lats, method='linear')